ESERCIZIO

Migrazioen da LSTM a GRU

Obbiettivo: misurare il guadagno computazionle e la stabilità della GRU in un task di classificazione
Refactoring: prendi un modello esistente basato su LSTM per la classificazione di testi (4 classi) e sostituisci il layer ricorrente con una GRU di pari unità (64)
Performance: confronta il tempo medio di esecuzione per epoca e il numero totale di parametri addestrabili mostrati da 'model.summary()'
Verifica: contralla se l'accuratezza finale sul test rimane comparabile a quella della LSTM
Sfida: prova a ridurre ulteriormente il learning rate e osserva se la GRU converge più velocemente verso un minimo stabile rispetto alla configurazione precedente.

In [2]:
"""
================================================================================
MIGRAZIONE DA LSTM A GRU - BENCHMARK REALE (DATASET AG NEWS)
================================================================================
CONFRONTO: LSTM vs GRU
DATASET: AG News (Classificazione news: 4 classi)
BACKEND: Keras 3 (PyTorch)
================================================================================
"""

import os
import time
import numpy as np

# Impostiamo il backend Keras
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers

# --- CARICAMENTO DATI (AG NEWS) ---
def load_ag_news(subset_size=10000):
    """
    Carica il dataset AG News tramite Hugging Face Datasets.
    Se non presente, lo installa al volo.
    """
    try:
        from datasets import load_dataset
    except ImportError:
        print("Installazione di 'datasets' necessaria...")
        os.system("pip install datasets")
        from datasets import load_dataset

    print("Caricamento dataset AG News...")
    dataset = load_dataset("fancyzhx/ag_news")
    
    # Prendiamo un subset per velocizzare il benchmark
    train_data = dataset["train"].shuffle(seed=42).select(range(subset_size))
    test_data = dataset["test"].shuffle(seed=42).select(range(int(subset_size * 0.2)))
    
    X_train = [item["text"] for item in train_data]
    y_train = [item["label"] for item in train_data]
    
    X_test = [item["text"] for item in test_data]
    y_test = [item["label"] for item in test_data]
    
    # One-hot encoding
    y_train = keras.utils.to_categorical(y_train, num_classes=4)
    y_test = keras.utils.to_categorical(y_test, num_classes=4)
    
    class_names = ["World", "Sports", "Business", "Sci/Tech"]
    return np.array(X_train), y_train, np.array(X_test), y_test, class_names

# --- ARCHITETTURA MODELLO ---

def build_benchmark_model(model_type, vocab_size, max_len, learning_rate=0.001):
    inputs = layers.Input(shape=(max_len,))
    x = layers.Embedding(input_dim=vocab_size, output_dim=64)(inputs)
    
    if model_type == "LSTM":
        # LSTM: 64 unità, 4 gate -> più parametri
        rec_layer = layers.LSTM(64)(x)
    else:
        # GRU: 64 unità, 3 gate (reset, update, new memory) -> meno parametri
        rec_layer = layers.GRU(64)(x)
    
    x = layers.BatchNormalization()(rec_layer)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(4, activation="softmax")(x)
    
    model = keras.Model(inputs, outputs, name=f"Model_{model_type}")
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# --- ESECUZIONE ESPERIMENTO ---

def run_experiment():
    # 1. Preparazione Dati
    X_train_raw, y_train, X_test_raw, y_test, class_names = load_ag_news(subset_size=8000)
    
    max_features = 10000 
    max_len = 50 # Lunghezza media news
    
    vectorize_layer = layers.TextVectorization(
        max_tokens=max_features,
        output_sequence_length=max_len
    )
    vectorize_layer.adapt(X_train_raw)
    
    X_train = vectorize_layer(X_train_raw)
    X_test = vectorize_layer(X_test_raw)
    
    results = {}

    # 2. Test LSTM
    print("\n" + "="*40)
    print("TEST 1: LSTM (Baseline)")
    print("="*40)
    model_lstm = build_benchmark_model("LSTM", max_features, max_len)
    model_lstm.summary()
    
    # Callback per salvare i migliori pesi sulla validazione
    checkpoint_lstm = keras.callbacks.ModelCheckpoint(
        filepath="best_lstm.weights.h5",
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max"
    )
    
    epochs = 10
    start = time.time()
    history_lstm = model_lstm.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=64, 
        validation_split=0.1, 
        verbose=1,
        callbacks=[checkpoint_lstm]
    )
    duration_lstm = time.time() - start
    
    # Ripristiniamo i pesi migliori prima del test finale
    model_lstm.load_weights("best_lstm.weights.h5")
    
    acc_lstm = model_lstm.evaluate(X_test, y_test, verbose=0)[1]
    results["LSTM"] = {"params": model_lstm.count_params(), "time": duration_lstm/epochs, "acc": acc_lstm}

    # 3. Test GRU
    print("\n" + "="*40)
    print("TEST 2: GRU (Migrazione)")
    print("="*40)
    model_gru = build_benchmark_model("GRU", max_features, max_len)
    model_gru.summary()
    
    # Callback per salvare i migliori pesi sulla validazione
    checkpoint_gru = keras.callbacks.ModelCheckpoint(
        filepath="best_gru.weights.h5",
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max"
    )
    
    start = time.time()
    history_gru = model_gru.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=64, 
        validation_split=0.1, 
        verbose=1,
        callbacks=[checkpoint_gru]
    )
    duration_gru = time.time() - start
    
    # Ripristiniamo i pesi migliori prima del test finale
    model_gru.load_weights("best_gru.weights.h5")
    
    acc_gru = model_gru.evaluate(X_test, y_test, verbose=0)[1]
    results["GRU"] = {"params": model_gru.count_params(), "time": duration_gru/epochs, "acc": acc_gru}

    # 4. Challenge: GRU con LR ridotto
    print("\n" + "="*40)
    print("TEST 3: GRU (Learning Rate 0.0005)")
    print("="*40)
    model_gru_stabile = build_benchmark_model("GRU", max_features, max_len, learning_rate=0.0005)
    
    # Callback per salvare i migliori pesi sulla validazione
    checkpoint_stabile = keras.callbacks.ModelCheckpoint(
        filepath="best_gru_stabile.weights.h5",
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max"
    )
    
    history_gru_stabile = model_gru_stabile.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=64, 
        validation_split=0.1, 
        verbose=1,
        callbacks=[checkpoint_stabile]
    )
    
    # Ripristiniamo i pesi migliori prima del test finale
    model_gru_stabile.load_weights("best_gru_stabile.weights.h5")
    
    acc_gru_stabile = model_gru_stabile.evaluate(X_test, y_test, verbose=0)[1]

    # --- REPORT FINALE ---
    print("\n" + "#"*50)
    print("REPORT DI MIGRAZIONE: LSTM vs GRU")
    print("#"*50)
    print(f"{'Metrica':<25} | {'LSTM':<12} | {'GRU':<12}")
    print("-" * 55)
    print(f"{'Parametri Totali':<25} | {results['LSTM']['params']:<12} | {results['GRU']['params']:<12}")
    print(f"{'Tempo per Epoca (s)':<25} | {results['LSTM']['time']:<12.2f} | {results['GRU']['time']:<12.2f}")
    print(f"{'Accuratezza Finale':<25} | {results['LSTM']['acc']:<12.2%} | {results['GRU']['acc']:<12.2%}")
    print("-" * 55)
    print(f"Guadagno computazionale stimato: {((results['LSTM']['time'] - results['GRU']['time']) / results['LSTM']['time'])*100:.1f}%")
    print(f"Accuratezza GRU (LR ridotto): {acc_gru_stabile:.2%}")

if __name__ == "__main__":
    run_experiment()

Caricamento dataset AG News...


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\datasets--fancyzhx--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 862605.14 examples/s]



TEST 1: LSTM (Baseline)


Model: "Model_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 50, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 667,300 (2.55 MB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/10
  6/113 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.2552 - loss: 1.4097

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:887: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n = torch._VF.gru(


113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.2814 - loss: 1.3721 - val_accuracy: 0.2425 - val_loss: 2.0141
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.6018 - loss: 0.8234 - val_accuracy: 0.2587 - val_loss: 1.9825
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8764 - loss: 0.3465 - val_accuracy: 0.6062 - val_loss: 0.8738
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9431 - loss: 0.1731 - val_accuracy: 0.6438 - val_loss: 0.8597
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9664 - loss: 0.1091 - val_accuracy: 0.7462 - val_loss: 0.7702
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9779 - loss: 0.0717 - val_accuracy: 0.7163 - val_loss: 1.3314
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9876 - loss: 0.0447 - val_accuracy: 0.3900 - val_loss: 5.1796
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9889 - loss: 0.0384 - val_accuracy: 0.768